In [94]:
# This notebook will be used to run the regression models to look at annual growth
import sqlalchemy 
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from scipy import stats
import scikit_posthocs as sp
# import statsmodels.api as sm
import statsmodels.formula.api as smf # this is to run a lm like R

# company_df.to_sql(name = 'companies', con = engine, if_exists = 'append', index = False)


In [95]:
load_dotenv("../../.env")

mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_password = os.environ.get("MYSQL_PASSWORD")
mysql_database = os.environ.get("MYSQL_DATABASE")


#the f goes in front to embed variables
engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_database}")

with engine.connect() as conn:
    print("Connection successful")



Connection successful


In [ ]:
#I am importing the data from MySQL and creating the daily returns variable

daily_prices = pd.read_sql("SELECT ticker, date, adjusted_close FROM daily_prices ", engine)

daily_prices.head()

# grouping by ticker, I am only reading adjusted_close info
daily_prices['previous_return'] = daily_prices.groupby('ticker')['adjusted_close'].shift(1)
daily_prices['daily_return'] = (daily_prices['adjusted_close'] - daily_prices['previous_return']) / daily_prices['previous_return']
# Double brackets keep the column name and convert it to a DataFrame


daily_prices['time'] = daily_prices.groupby('ticker').cumcount() + 1
daily_prices.head(10)

,ticker,date,adjusted_close,previous_return,daily_return,time
0,DLR,2025-06-09,171.67,NaN,NaN,1.0
1,EQIX,2025-06-09,887.30,NaN,NaN,1.0
2,IRM,2025-06-09,97.79,NaN,NaN,1.0
3,DLR,2025-06-10,172.65,171.67,0.005709,2.0
4,EQIX,2025-06-10,887.16,887.30,-0.000158,2.0
5,IRM,2025-06-10,97.64,97.79,-0.001534,2.0
6,DLR,2025-06-11,170.86,172.65,-0.010368,3.0
7,EQIX,2025-06-11,873.83,887.16,-0.015025,3.0
8,IRM,2025-06-11,97.56,97.64,-0.000819,3.0
9,DLR,2025-06-12,171.53,170.86,0.003921,4.0


In [100]:

results_list = []


for ticker in daily_prices['ticker'].unique():
    temp_df = daily_prices[daily_prices['ticker'] == ticker]
    returnbytime = smf.ols("adjusted_close ~ time", data=temp_df)
    results = returnbytime.fit()
    results_list.append({
        'ticker': ticker,
        'regression_slope' : results.params['time'],
        'r_squared':  results.rsquared,
       # 'p_value_slope': results.pvalues['time'],
        #'p_value_f': results.f_pvalue,
        'intercept': results.params['Intercept']
            })
    print(f'\n\n the ticker is {ticker}', results.summary())
    
results_df = pd.DataFrame(results_list)
results_df.head()

results_df.to_sql(name = 'regression_results', con = engine, if_exists = 'append', index = False)




 the ticker is DLR                             OLS Regression Results                            
Dep. Variable:         adjusted_close   R-squared:                       0.294
Model:                            OLS   Adj. R-squared:                  0.291
Method:                 Least Squares   F-statistic:                     103.6
Date:                Wed, 10 Jun 2026   Prob (F-statistic):           1.45e-20
Time:                        09:57:46   Log-Likelihood:                -949.92
No. Observations:                 251   AIC:                             1904.
Df Residuals:                     249   BIC:                             1911.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    159.6996      1.35

3

In [98]:
#Running the Kruskal-wallis test


# # Stability (Daily Range)
# #need the daily ranges (close-open)

# cv_data = pd.read_sql("SELECT ticker, cv_daily_range FROM monthly_summary", engine)

# # These are not statistically different from each other. Must interpret tomorrow
# # stats.kruskal(cv_data[cv_data['ticker'] == 'DLR']['cv_daily_range'], 
# #               cv_data[cv_data['ticker'] == 'EQIX']['cv_daily_range'], 
# #               cv_data[cv_data['ticker'] == 'IRM']['cv_daily_range'])

# # sp.posthoc_dunn(cv_data, 'cv_daily_range', 'ticker', p_adjust='bonferroni')

# # cv_data.groupby('ticker')['cv_daily_range'].median()
# stats.kruskal(daily_prices[daily_prices['ticker'] == 'DLR']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'EQIX']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'IRM']['daily_returns'].dropna())


# Growth (Daily Returns)


ProgrammingError: (mysql.connector.errors.ProgrammingError) 1054 (42S22): Unknown column 'nan' in 'field list'
[SQL: UPDATE daily_prices SET daily_return = %(dr)s WHERE ticker = %(t)s AND `date` = %(d)s]
[parameters: {'dr': nan, 't': 'DLR', 'd': '2025-06-09'}]
(Background on this error at: https://sqlalche.me/e/20/f405)